# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This is a binary classification task: for each page, predict whether it will "churn" (trend_direction == "down") or not. I'm not doing clustering or ranking here at the core — classification is the right fit because the outcome is a clear yes/no label (churned vs. not churned) that I can validate against real observed outcomes in the data.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "pages,", df.shape[1], "columns")
print(df["trend_direction"].value_counts())

30000 pages, 44 columns
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_churned = 1 if trend_direction == "down", else 0. This label comes from a defined rule inside the current data window, not a true future outcome — it's a proxy, since "down" describes the recent trend rather than something that happens strictly after a decision point. A stronger version (for later weeks) would define churn as a future decline over the next N days, using only signals known before that window.

In [2]:
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Churn rate:", round(df["is_churned"].mean(), 3))

Churn rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use Precision@50: of the top 50 pages my model flags as highest churn-risk, what fraction actually churned? This is the right metric because reviewers have limited capacity — they can only check a handful of pages, so what matters is whether the top of the list is right, not overall accuracy across all 30,000 pages.

In [3]:
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()
print("Metric defined: precision_at_k(scores, labels, k)")

Metric defined: precision_at_k(scores, labels, k)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page. Each row already represents a single page with its own visibility, freshness, and trend signals — that's the natural grain for this task, since the reviewer's decision ("should I check this page?") happens at the page level, not the client or query level.

In [4]:
df[["content_id", "impressions_90d", "days_since_last_update", "trend_direction", "is_churned"]].head(10)

,content_id,impressions_90d,days_since_last_update,trend_direction,is_churned
0,content_304f48230142,3803,20,down,1
1,content_a1fb4e703a9e,15320,25,down,1
2,content_9aa793d4d895,12581,20,down,1
3,content_331d6c4de07b,11751,22,stable,0
4,content_d99b7a2d90ca,19140,14,down,1
5,content_d4084a4bc775,3970,20,down,1
6,content_9a34b442b552,20,20,down,1
7,content_a63219c6e95a,1724,22,stable,0
8,content_5e6c160719bc,32574,20,down,1
9,content_c27558df2b0c,1240,104,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The fixed rule I tested in ML-02 (stale + visible) only caught 35 out of 30,000 pages — far too narrow to be useful at scale, even though it was fairly precise (74% churned). Real churn risk likely depends on many signals interacting together (position, CTR, word count, age, engagement) in ways a simple AND-rule can't capture. A model can weigh all these signals jointly and rank the full 30,000 pages, not just the tiny sliver a hand rule flags.

In [5]:
print("Hand rule coverage: 35 of 30000 pages (0.1%) — too narrow for practical use")
print("A model can score and rank ALL pages using many signals jointly")

Hand rule coverage: 35 of 30000 pages (0.1%) — too narrow for practical use
A model can score and rank ALL pages using many signals jointly


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.